In [1]:
import os
os.chdir('/Users/angelinagupta/Desktop/sem 6/biomedical-nlp-project-beta')

In [2]:
import pandas as pd
import numpy as np
import json
import os
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report

## 1. Load NCBI Disease Dataset
Load from the saved CSV files in `data/` (created in notebook 01). Falls back to HuggingFace parquet if CSVs are missing.

In [3]:
import ast
import re

def parse_col(series):
    """Parse a column containing Python or numpy-style stringified lists.

    The CSVs were saved from numpy arrays, so columns look like:
      ner_tags: '[0 0 0 1 2]'          — numpy int array, no commas
      tokens:   "['tok1' 'tok2' ...]"  — numpy str array, no commas

    IMPORTANT: ast.literal_eval MUST NOT be tried first on numpy string arrays
    because Python treats adjacent string literals as implicit concatenation,
    returning a single-element list instead of the full token list.
    """
    if len(series) == 0:
        return series
    first = series.iloc[0]
    if not isinstance(first, str):
        return series

    def parse_one(s):
        s = s.strip()
        inner = s[1:-1].strip()   # strip outer [ ]
        if not inner:
            return []
        # 1. Numpy integer array: [0 0 1 2]  (no commas, all numeric)
        try:
            return [int(x) for x in inner.split()]
        except ValueError:
            pass
        # 2. Numpy string array: ['tok1' 'tok2' ...]  (no commas, quoted)
        items = re.findall(r"'([^']*)'", inner)
        if items:
            return items
        # 3. Standard Python list: ['a', 'b'] or [0, 1, 2]  (commas present)
        try:
            return ast.literal_eval(s)
        except (ValueError, SyntaxError):
            pass
        # 4. Last resort: whitespace split
        return inner.split()

    return series.apply(parse_one)

train_path = 'data/ncbi_train.csv'
test_path  = 'data/ncbi_test.csv'

if os.path.exists(train_path) and os.path.exists(test_path):
    df_train = pd.read_csv(train_path)
    df_test  = pd.read_csv(test_path)
    df_train['tokens']   = parse_col(df_train['tokens'])
    df_train['ner_tags'] = parse_col(df_train['ner_tags'])
    df_test['tokens']    = parse_col(df_test['tokens'])
    df_test['ner_tags']  = parse_col(df_test['ner_tags'])
    print('Loaded from local CSVs.')
else:
    PARQUET_BASE = 'hf://datasets/ncbi_disease@refs/convert/parquet/ncbi_disease'
    df_train = pd.read_parquet(f'{PARQUET_BASE}/train/0000.parquet')
    df_test  = pd.read_parquet(f'{PARQUET_BASE}/test/0000.parquet')
    print('Loaded from HuggingFace parquet.')

print(f'Train sentences : {len(df_train)}')
print(f'Test  sentences : {len(df_test)}')
print(f'Tokens sample   : {df_test["tokens"].iloc[0][:5]}')
print(f'Tags  sample    : {df_test["ner_tags"].iloc[0][:5]}')

Loaded from local CSVs.
Train sentences : 5433
Test  sentences : 941
Tokens sample   : ['Clustering', 'of', 'missense', 'mutations', 'in']
Tags  sample    : [0, 0, 0, 0, 0]


## 2. Build Disease Dictionary
Extract every annotated disease entity from the training set (joining B-Disease / I-Disease token sequences). Store both original-case and lowercased versions.

In [4]:
label_map = {0: 'O', 1: 'B-Disease', 2: 'I-Disease'}

disease_dict = set()

for _, row in df_train.iterrows():
    tokens = row['tokens']
    tags   = row['ner_tags']

    current_entity = []
    for token, tag in zip(tokens, tags):
        if tag == 1 or tag == 'B-Disease':
            if current_entity:
                ent = ' '.join(current_entity)
                disease_dict.add(ent)
                disease_dict.add(ent.lower())
            current_entity = [token]
        elif tag == 2 or tag == 'I-Disease':
            current_entity.append(token)
        else:
            if current_entity:
                ent = ' '.join(current_entity)
                disease_dict.add(ent)
                disease_dict.add(ent.lower())
                current_entity = []
    if current_entity:
        ent = ' '.join(current_entity)
        disease_dict.add(ent)
        disease_dict.add(ent.lower())

print(f'Total unique entities collected in dictionary: {len(disease_dict)}')
print('Sample entries:', list(disease_dict)[:10])

Total unique entities collected in dictionary: 2233
Sample entries: ['familial neurohypophyseal diabetes insipidus', 'CHRPEs', 'MEF', 'sub - total deficiency of C6 and C7', 'neurological disturbance', 'inner ear abnormality', 'severe acroparesthesia', 'x - linked glucose - 6 - phosphate dehydrogenase deficiency', 'Endocrine neoplasms', 'polyposis']


## 3. Medical Suffix Rules
Words ending in these suffixes are flagged as potential diseases when no dictionary match is found.

In [5]:
MEDICAL_SUFFIXES = ('itis', 'osis', 'emia', 'oma', 'pathy', 'plasia', 'trophy', 'ectomy')

def has_medical_suffix(token):
    return any(token.lower().endswith(suf) for suf in MEDICAL_SUFFIXES)

# Quick sanity check
test_words = ['hepatitis', 'fibrosis', 'leukemia', 'carcinoma', 'neuropathy',
               'hyperplasia', 'atrophy', 'appendectomy', 'patient', 'gene']
for w in test_words:
    print(f'{w:20s}  suffix match: {has_medical_suffix(w)}')

hepatitis             suffix match: True
fibrosis              suffix match: True
leukemia              suffix match: True
carcinoma             suffix match: True
neuropathy            suffix match: True
hyperplasia           suffix match: True
atrophy               suffix match: True
appendectomy          suffix match: True
patient               suffix match: False
gene                  suffix match: False


## 4. Rule-Based NER Function
- Check 4-word → 3-word → 2-word → 1-word windows (longest match wins)
- Dictionary lookup is case-insensitive
- Suffix rule fires only if no dictionary match was found for that token
- Outputs BIO tags, one per input token

In [6]:
def extract_entities_rule_based(tokens):
    """Return BIO tag list for a list of tokens."""
    n = len(tokens)
    preds = ['O'] * n
    i = 0

    while i < n:
        matched = False

        # Longest-match dictionary scan (4 → 1 tokens)
        for window in range(min(4, n - i), 0, -1):
            span = tokens[i : i + window]
            span_str = ' '.join(span)
            if span_str in disease_dict or span_str.lower() in disease_dict:
                preds[i] = 'B-Disease'
                for k in range(1, window):
                    preds[i + k] = 'I-Disease'
                i += window
                matched = True
                break

        # Suffix rule fallback (single token only)
        if not matched:
            if has_medical_suffix(tokens[i]):
                preds[i] = 'B-Disease'
            i += 1

    return preds

## 5. Evaluate on Test Set

In [8]:
# Filter out any rows where tokens and ner_tags lengths don't match
df_test = df_test[
    df_test.apply(lambda row: len(row['tokens']) == len(row['ner_tags']), axis=1)
].reset_index(drop=True)

print(f"Clean test rows: {len(df_test)}")

y_true = []
y_pred = []

for _, row in df_test.iterrows():
    tokens = row['tokens']
    tags   = row['ner_tags']

    true_tags = [
        label_map.get(t, 'O') if isinstance(t, (int, np.integer)) else t
        for t in tags
    ]
    pred_tags = extract_entities_rule_based(tokens)

    # Extra safety check
    if len(true_tags) != len(pred_tags):
        continue

    y_true.append(true_tags)
    y_pred.append(pred_tags)

# Token-level accuracy
total   = sum(len(t) for t in y_true)
correct = sum(
    t == p
    for true_seq, pred_seq in zip(y_true, y_pred)
    for t, p in zip(true_seq, pred_seq)
)
print(f'Token-Level Accuracy: {correct / total:.4f}\n')

prec = precision_score(y_true, y_pred)
rec  = recall_score(y_true, y_pred)
f1   = f1_score(y_true, y_pred)

print('Entity-Level Metrics (seqeval):')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1-Score  : {f1:.4f}\n')

print('Detailed Classification Report:')
print(classification_report(y_true, y_pred))

Clean test rows: 940
Token-Level Accuracy: 0.9453

Entity-Level Metrics (seqeval):
  Precision : 0.5378
  Recall    : 0.6225
  F1-Score  : 0.5771

Detailed Classification Report:
              precision    recall  f1-score   support

     Disease       0.54      0.62      0.58       959

   micro avg       0.54      0.62      0.58       959
   macro avg       0.54      0.62      0.58       959
weighted avg       0.54      0.62      0.58       959



## 6. Example Predictions vs Ground Truth
Show 10 sentences where the rule-based system makes errors. Mismatched tokens are highlighted.

In [9]:
print('10 Example Sentences — Predictions vs Ground Truth\n')
print('Legend: [token] TRUE->PRED  (only mismatched tokens shown in detail)\n')

shown = 0
for i, (true_seq, pred_seq) in enumerate(zip(y_true, y_pred)):
    if true_seq == pred_seq:
        continue  # skip perfectly correct sentences

    tokens = df_test.iloc[i]['tokens']
    print(f'--- Sentence {i} ---')

    # Print full sentence with BIO annotation; mark mismatches
    parts = []
    for tok, t, p in zip(tokens, true_seq, pred_seq):
        if t != p:
            parts.append(f'**{tok}**[TRUE={t}|PRED={p}]')
        else:
            parts.append(f'{tok}[{t}]')
    print(' '.join(parts))
    print()

    shown += 1
    if shown >= 10:
        break

10 Example Sentences — Predictions vs Ground Truth

Legend: [token] TRUE->PRED  (only mismatched tokens shown in detail)

--- Sentence 0 ---
Clustering[O] of[O] missense[O] mutations[O] in[O] the[O] ataxia[B-Disease] -[I-Disease] telangiectasia[I-Disease] gene[O] in[O] a[O] **sporadic**[TRUE=B-Disease|PRED=O] **T**[TRUE=I-Disease|PRED=O] **-**[TRUE=I-Disease|PRED=O] **cell**[TRUE=I-Disease|PRED=O] **leukaemia**[TRUE=I-Disease|PRED=B-Disease] .[O]

--- Sentence 1 ---
Ataxia[B-Disease] -[I-Disease] telangiectasia[I-Disease] ([O] A[B-Disease] -[I-Disease] T[I-Disease] )[O] is[O] a[O] **recessive**[TRUE=B-Disease|PRED=O] **multi**[TRUE=I-Disease|PRED=O] **-**[TRUE=I-Disease|PRED=O] **system**[TRUE=I-Disease|PRED=O] **disorder**[TRUE=I-Disease|PRED=O] caused[O] by[O] mutations[O] in[O] the[O] ATM[O] gene[O] **at**[TRUE=O|PRED=B-Disease] 11q22[O] -[O] q23[O] ([O] ref[O] .[O] 3[O] )[O] .[O]

--- Sentence 2 ---
The[O] risk[O] of[O] cancer[B-Disease] ,[O] especially[O] **lymphoid**[TRUE=B-Disea

## 7. Error Analysis

### Why does dictionary lookup fail?

1. **Unseen entities (OOV)** — The most dominant failure mode. Any disease in the test set that didn't appear in the training text is invisible to the dictionary. The NCBI dataset has ~2 100 unique entities; many appear only once, guaranteeing test-set misses.

2. **Abbreviations and acronyms** — Biomedical text is dense with abbreviations (APC, BRCA1, DMD, HD). These are frequently annotated as diseases but carry no medical suffix and only appear in the dictionary if the exact same abbreviation was in training data. Even a single new abbreviation defeats the rule.

3. **Partial / fragmented matches** — The greedy longest-match algorithm can fail when a 4-word span almost matches a dictionary entry but differs by one token (e.g., a parenthetical insertion). The system then tries shorter windows and may tag only a fragment as `B-Disease`, leaving the rest as `O`.

4. **Context-dependent ambiguity** — Words like *tumor*, *cancer*, *syndrome* are diseases in specific contexts but also appear as generic modifiers ("tumor suppressor gene"). A dictionary always fires on the noun regardless of its syntactic role, causing false positives that hurt precision.

5. **Tokenization mismatches** — Hyphenated forms ("ataxia-telangiectasia" vs. "ataxia - telangiectasia"), possessives, and trailing punctuation mean the reconstructed span string won't match the dictionary entry even when the same disease appears in both splits.

6. **No morphological generalisation** — The dictionary is exact-match. "Lymphomas" and "lymphoma" are treated as distinct entries. Without lemmatization or fuzzy matching, plural/inflected forms of known diseases are missed.

### Why the suffix rule still under-performs
Many multi-word diseases (e.g., "familial adenomatous polyposis") only have a suffix hint in the *last* token; the prefix tokens are missed. Conversely, many gene/protein names end in *-oma* or *-itis* but are not diseases, inflating false positives.

## 8. Save Baseline Results

In [10]:
os.makedirs('results', exist_ok=True)
results_path = 'results/ner_results.json'

new_result = {
    'method'   : 'rule_based',
    'precision': round(prec, 4),
    'recall'   : round(rec,  4),
    'f1'       : round(f1,   4)
}

results_list = []
if os.path.exists(results_path):
    with open(results_path) as f:
        try:
            results_list = json.load(f)
        except json.JSONDecodeError:
            pass

# Upsert: replace existing rule_based entry if present
updated = False
for r in results_list:
    if r.get('method') == 'rule_based':
        r.update(new_result)
        updated = True
        break
if not updated:
    results_list.append(new_result)

with open(results_path, 'w') as f:
    json.dump(results_list, f, indent=4)

print('Baseline results saved to results/ner_results.json')
print(json.dumps(new_result, indent=4))

Baseline results saved to results/ner_results.json
{
    "method": "rule_based",
    "precision": 0.5378,
    "recall": 0.6225,
    "f1": 0.5771
}
